#TITLE:

Advanced Dimensionality Reduction & Data Augmentation for Image-Based ML
With Fashion MNIST — PCA, t-SNE, UMAP, Albumentations, and Classifier

# 🧾 HOW TO USE THIS IN COLAB

Open Google Colab

Start a new notebook

Enable GPU: Runtime → Change runtime type → Hardware Accelerator: GPU

Copy and paste each code block from this .docx file into a new Colab cell

Run the cells and modify as needed!

# 🛠️ INSTALL DEPENDENCIES


In [ ]:
!pip install -q umap-learn albumentations opentsne torchvision


In [2]:
# 🔧 IMPORTS & SETUP
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import torch
import torchvision
from torchvision import transforms
from tensorflow.keras.datasets import fashion_mnist
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import umap.umap_ as umap
from opentsne import TSNE as OpenTSNE
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

import warnings
warnings.filterwarnings("ignore")
sns.set(style='whitegrid')


In [ ]:
# 📥 LOAD AND PREPROCESS DATA
(X_train, y_train), (X_test, y_test) = fashion_mnist.load_data()
X = X_train.reshape((X_train.shape[0], -1))
X = StandardScaler().fit_transform(X)
y = y_train

label_dict = {
    0: "T-shirt", 1: "Trouser", 2: "Pullover", 3: "Dress", 4: "Coat",
    5: "Sandal", 6: "Shirt", 7: "Sneaker", 8: "Bag", 9: "Ankle boot"
}


In [ ]:
# 📊 PCA: Principal Component Analysis
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X)
print("Explained Variance Ratio:", pca.explained_variance_ratio_)


In [ ]:
# 📈 Visualize
def plot_2D(X_emb, y, title):
    df = pd.DataFrame({'x': X_emb[:, 0], 'y': X_emb[:, 1], 'label': y})
    plt.figure(figsize=(10, 7))
    sns.scatterplot(data=df, x='x', y='y', hue='label', palette='tab10', alpha=0.7)
    plt.title(title)
    plt.legend()
    plt.show()

plot_2D(X_pca, y, "PCA - 2D Visualization")


In [ ]:
# 🧪 Scree Plot
pca_full = PCA().fit(X)
plt.figure(figsize=(8, 5))
plt.plot(np.cumsum(pca_full.explained_variance_ratio_), marker='o')
plt.title("Scree Plot")
plt.xlabel("Components")
plt.ylabel("Cumulative Explained Variance")
plt.grid(True)
plt.show()


In [ ]:
# 🌌 t-SNE with openTSNE (GPU-Friendly)
X_sample = X[:5000]
y_sample = y[:5000]

tsne = OpenTSNE(n_components=2, perplexity=30, n_jobs=4)
X_tsne = tsne.fit(X_sample)

plot_2D(X_tsne, y_sample, "t-SNE - 2D Projection")


In [ ]:
# 🌐 UMAP
reducer = umap.UMAP(n_neighbors=15, min_dist=0.1)
X_umap = reducer.fit_transform(X_sample)

plot_2D(X_umap, y_sample, "UMAP - 2D Projection")


In [ ]:
# 🧪 DATA AUGMENTATION
from PIL import Image
import albumentations as A
import cv2

# Sample images for augmentation
sample_images = X_train[:5]
pil_images = [Image.fromarray(img) for img in sample_images]

# TorchVision transforms
transform = transforms.Compose([
    transforms.RandomHorizontalFlip(p=1.0),
    transforms.RandomRotation(30),
    transforms.RandomResizedCrop(28, scale=(0.8, 1.0)),
    transforms.ColorJitter(brightness=0.5, contrast=0.5)
])

def show_augmented(images, transform_fn, title):
    fig, axs = plt.subplots(1, len(images), figsize=(15, 3))
    for i, img in enumerate(images):
        axs[i].imshow(transform_fn(img), cmap='gray')
        axs[i].axis('off')
    plt.suptitle(title)
    plt.show()

show_augmented(pil_images, transform, "Torchvision Basic Augmentations")


In [ ]:
# 🧪 Gaussian Noise
def add_gaussian_noise(img, std=0.2):
    noise = np.random.normal(0, std, (28, 28))
    noisy = np.clip(np.array(img)/255.0 + noise, 0, 1)
    return Image.fromarray((noisy * 255).astype(np.uint8))

show_augmented(pil_images, add_gaussian_noise, "Gaussian Noise Augmentation")


In [ ]:
# 🧪 Albumentations Pipeline
alb_transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.Rotate(limit=40, p=0.8),
    A.RandomBrightnessContrast(p=0.5),
    A.ShiftScaleRotate(shift_limit=0.0625, scale_limit=0.2, rotate_limit=15, p=0.8),
    A.GaussianBlur(p=0.2)
])

def apply_albumentations(images, transform):
    fig, axs = plt.subplots(1, len(images), figsize=(15, 3))
    for i, img in enumerate(images):
        aug = transform(image=np.array(img))["image"]
        axs[i].imshow(aug, cmap='gray')
        axs[i].axis('off')
    plt.suptitle("Albumentations – Advanced Augmentations")
    plt.show()

apply_albumentations(pil_images, alb_transform)


In [ ]:
# ✅ CLASSIFIER ON REDUCED FEATURES
X_reduced = PCA(n_components=50).fit_transform(X[:10000])
y_reduced = y[:10000]

clf = LogisticRegression(max_iter=2000)
score = cross_val_score(clf, X_reduced, y_reduced, cv=5)
print("Accuracy (PCA-50):", score.mean())


#QUESTIONS
## 🟦 PCA

### Coding:

Train a classifier using PCA(100) components.

Compare performance with PCA(10).

Visualize PCA-transformed images in 3D.

### Theory:

What does PCA optimize?

When can PCA hurt model performance?

## 🌌 t-SNE

### Coding:

Try different perplexities (5, 30, 50).

Apply t-SNE on PCA-reduced data.

Use t-SNE for only two classes.

### Theory:

Why is t-SNE bad for classification?

What does perplexity control?

## 🌐 UMAP

### Coding:

Change n_neighbors and observe changes.

Use UMAP output for KMeans clustering.

Compare UMAP with PCA visually.

### Theory:

How does UMAP preserve global structure?

Why is UMAP faster than t-SNE?

## 🧪 Augmentation

### Coding:

Build a pipeline with 4 augmentations.

Apply augmentation only to class 9.

Show a grid of 10 random augmentations.

### Theory:

What are the dangers of over-augmentation?

When to use GAN-based augmentation?